Transfer learning is a method where we use a pre-trained neural network instead of training a model from scratch. Powerful models like ResNet, AlexNet, InceptionNet, or DenseNet are already trained on very large datasets such as ImageNet, which contains 1000 classes. These networks have already learned important low-level features like edges, lines, textures, and simple shapes in their early layers.

In transfer learning, we take one of these pre-trained models and freeze its layers so their learned weights do not change. Then we remove the final classification layer (which was originally designed to output 1000 classes) and replace it with a new final layer that matches our own task. For example, if we want to classify dogs, cats, cows, and sheep, we would modify the final layer to output only the number of classes we need (for example, 4 or 5). After that, we train only this new final layer on our dataset. Since the earlier layers already know how to extract useful image features, we only need to retrain the last layer to adapt the model to our specific classes. This makes training faster, requires less data, and usually gives very good results.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torchvision
from torch.autograd import Variable
from torchvision import datasets, models, transforms
import os
import numpy as np

# Data augmentation and normalization for training

In our training pipeline, we apply several transforms to prepare the images before feeding them into the neural network.  

The goal is:

1. Perform **data augmentation**
2. Ensure **all images are 224 × 224**
3. Convert images into **PyTorch tensors**
4. Normalize them for stable training

---

## 1️⃣ RandomResizedCrop(224)

This is the first data augmentation step.

### What it does:

- It **randomly crops** a portion of the image.
- The crop size is between **8% (0.08) and 100% (1.0)** of the original image area.
- The aspect ratio is randomly chosen between **3/4 and 4/3**.
- After cropping, the image is resized to **224 × 224** (because our model ResNet expect images of this size) using **bilinear interpolation (default)**.


- It ensures the **final image size is always 224 × 224** for all images

---

## 2️⃣ RandomHorizontalFlip()

This randomly flips the image horizontally.

- It keeps default probability (usually 50%).


## 3️⃣ ToTensor()

PyTorch works with **tensors**, not raw images.

This transform:

- Converts image pixel values from `[0, 255]`
- To floating-point values in `[0, 1]`
- Changes shape from:
  
  `H × W × C  →  C × H × W`

Without this step, PyTorch cannot process the image.

---

## 4️⃣ Normalize(mean, std)

This is one of the most important steps.

We normalize each channel using the formula:

$$
\text{input[channel]} = \frac{\text{input[channel]} - \text{mean[channel]}}{\text{std[channel]}}
$$
### For RGB images:

We have 3 channels:
- R (Red)
- G (Green)
- B (Blue)

So mean and std must be sequences of 3 values.

Example (ImageNet statistics):
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]


These represent:
- Mean of R, G, B channels
- Standard deviation of R, G, B channels



## 📊 Where Do Mean and Std Come From?

Ideally:

- You loop through your dataset
- Use `.mean()` and `.std()` in PyTorch
- Calculate statistics for each channel

However:
- Most natural image datasets have values close to ImageNet statistics
- So we often reuse these values

---

## 🔥 Why Training and Validation Transforms Differ

Training:
- RandomResizedCrop
- RandomHorizontalFlip
- Adds randomness

Validation:
- Resize
- CenterCrop
- No randomness

We want:
- **Randomness for training**
- **Consistency for evaluation**

---


Final output:
- Tensor
- Shape: `3 × 224 × 224`
- Properly normalized
- Ready for CNN input


In [ ]:

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}